In [ ]:
!pip install xlsxwriter
import pandas as pd
from pandas.core.common import flatten
import json
import math
import numpy as np

# Drawing

In [2]:
def read_jsons(filename='None',address='None'):
    with open(address+filename,'r') as file:
        loaded = json.load(file)

    return loaded


def read_excel_file(filename='None', address='None'):
    df_from_file=pd.read_excel(address+filename)

    return df_from_file


def reset_row_numbers(dataframe: pd.DataFrame, column_name: str, idx_length: int, drop_old_index: bool):
    dataframe[column_name] = range(1,idx_length+1)
    return dataframe.reset_index(drop=drop_old_index)


def extract_columns(dataframe: pd.DataFrame, columns: list):
    extracted = dataframe.loc[:, columns].copy()

    return extracted


def customize_columns(dataframe: pd.DataFrame, table_name: str, changes: list, output: list):
    if changes == []:
        return dataframe

    elif 'clearance' in table_name:
        len_df = len(dataframe)
        init_df = dataframe.to_dict(orient='list')

        return new_strip_columns([init_df, len_df], '--', changes, output)

    elif 'tkii' in table_name:
        len_df = len(dataframe)
        init_df = new_copied_columns(dataframe, changes, 'WAKTU TIBA', output)
        return init_df

    elif 'empty' in table_name:
        len_df = len(dataframe)
        init_df = column_renamer(dataframe, table_name)
        # print(init_df.columns)
        init_df = new_copied_columns(init_df, changes, 'WAKTU TIBA', output)
        # print(init_df.columns)

        return  init_df


def new_strip_columns(material_list: list, char: any, changes: list, output: list):
    dictionary = material_list[0]
    len_dict = material_list[1]
    for key in changes:
            # print(key)
            dictionary[key] = [char]*len_dict

    # print(dictionary.keys())
    new_df = pd.DataFrame.from_dict(dictionary)
    return new_df[output]


def new_copied_columns(dictionary: dict, changes: list, copy_from: str ,output: list):
    for item in changes:
        dictionary[item] = dictionary[copy_from]

    # print(dictionary.keys())

    return dictionary[output]

# Processing

In [3]:
def filter_row(dataframe: pd.DataFrame, name: str, filter: any):
    if filter == []:
        return dataframe
    else:
        dataframe = filter_operator(dataframe, name, filter)
        return dataframe


def filter_operator(dataframe: pd.DataFrame, name: str, filter: any):
    if not isinstance(filter, list):
        dataframe = filter_by_text(dataframe, filter)
    elif isinstance(filter, list) and len(filter) != 0:
        dataframe = filter_by_list(dataframe, name, filter)

    # print(dataframe.columns)
    dataframe = reset_row_numbers(dataframe, 'NO', len(dataframe), True)
    return dataframe


def filter_by_text(dataframe: pd.DataFrame, filter: any):
    if filter == 'GTminus':
        return dataframe.loc[dataframe['GT'] < 500].copy()
    elif filter == 'GTplus':
        return dataframe.loc[dataframe['GT'] >= 500].copy()
    elif filter == 'Empty':
        name_filter = start_with(dataframe)
        nihil_filter = lambda_nihil(dataframe)
        combined = combiner_sorter(name_filter, nihil_filter)
        combined.drop(columns=["MUATAN BONGKAR","MUATAN MUAT"], inplace=True)

        return combined


def filter_by_list(dataframe: pd.DataFrame, name: str, filter: any):
    # print(name)
    if 'tkii' in name:
        dataframe = dataframe.loc[dataframe[filter[0]].str.contains(filter[1], case=True)].copy()
        # dataframe = dataframe.drop(columns='LOKASI BONGKAR')
    elif 'domestic' in name or 'export' in name:
        dataframe = dataframe.loc[dataframe[filter[0]].str.contains(filter[1], case=True)].copy()
    elif 'load' in name:
        dataframe = dataframe.loc[(dataframe[filter[0]].str.contains(filter[2], case=True, regex=False)) | (dataframe[filter[1]].str.contains(filter[2], case=True, regex=False))].copy()
        # dataframe = dataframe.drop(columns=['JENIS BARANG','JENIS MUATAN'])

    return dataframe


def start_with(dataframe: pd.DataFrame):
    prefixes = ("TB.", "OB.", "TK.", "BG.", "AHTS.")
    return dataframe[dataframe['NAMA KAPAL'].str.startswith(prefixes, na=False)]


def lambda_nihil(dataframe: pd.DataFrame):
    is_nihil = lambda col: dataframe[col].astype(str).str.upper().str.strip() == "NIHIL"
    return dataframe[is_nihil("MUATAN BONGKAR") & is_nihil("MUATAN MUAT")]


def combiner_sorter(dataframe1, dataframe2):
    combined = pd.concat([dataframe1, dataframe2]).drop_duplicates()
    combined.sort_values(by="WAKTU TOLAK")
    combined = reset_row_numbers(combined, 'NO', len(combined), True)
    return combined


def rearrange_rows(dataframe: pd.DataFrame, name: str, keys: list, output: list):
    if name in ['sib_kecil','sib_besar','empty']:
        dict_form = dataframe.to_dict('list')
        return dict_form, len(dataframe)

    elif name in ['clearance']:
        dict_form = dataframe.to_dict('list')
        info_collect = mark_index_note_adds(dict_form[keys[0]])
        dict_form = modify(dict_form, name, keys, info_collect)
        first_key = next(iter(dict_form))
        return dict_form, len(dict_form[first_key])

    else:
        dict_form, len_data = special_rearranger(dataframe, name, keys, output)
        return dict_form, len_data


def special_rearranger(dataframe: pd.DataFrame, name: str, keys: list, output: list):
    if dataframe.empty:
        return {}, 0
    else:
        expanded_df = expand_register_dataframe(dataframe, keys)
        expanded_df = nat_error_fixer(expanded_df, output)
        expanded_df = column_dropper(expanded_df)
        expanded_df = column_renamer(expanded_df, name)
        # output_path = "RESULT_EXPANDED_FIRST_ONLY.xlsx"
        # expanded_df.to_excel(output_path, index=False)
        dict_form = expanded_df.to_dict('list')
        first_key = next(iter(dict_form))
        # print(dict_form)
        return dict_form, len(dict_form[first_key])


def mark_index_note_adds(dict_values: list):
    add_rows, marked_idx = [], []
    for index, item in enumerate(dict_values):
        # print(item)
        if type(item) == str and ';' in item:
            item = item.split('; ')
            marked_idx.append(index)
            add_rows.append(len(item))

    return {'indexes': marked_idx, 'adds': add_rows}


def modify(dictionary: dict, name:str, keys: list, information: dict):
    info_keys = list(information.keys())
    # print(information[info_keys[0]])
    # print(information[info_keys[1]])
    for key in dictionary.keys():
        if key not in keys:
            dictionary[key] = insert_nan(dictionary[key],information[info_keys[0]],information[info_keys[1]])
        elif key in keys:
            dictionary[key] = flatten_list(dictionary[key], name)

    return dictionary


def insert_nan(current_column: list, current_index: list, added_rows: list):
    new_list = []
    for idx in range(len(current_column)):
        if idx in current_index:
            at_idx = current_index.index(idx)
            # print('current index', idx, 'element of marked index at', at_idx)
            new_list.append(current_column[idx])
            for i in range(added_rows[at_idx]-1):
                new_list.append(None)
        elif idx not in current_index:
            new_list.append(current_column[idx])

    return new_list


def flatten_list(current_column: list, name: str):
    if name in ['clearance']:
        for idx in range(len(current_column)):
            if '; ' in str(current_column[idx]):
                current_column[idx] = current_column[idx].split('; ')

        current_column = list(flatten(current_column))
        return current_column


def expand_register_dataframe(df, multi_columns):
    all_rows = []
    for _, row in df.iterrows():
        all_rows.extend(expand_row_partial(row, multi_columns))
    return pd.DataFrame(all_rows)


def expand_row_partial(row, multi_columns):
    zipped_values = normalize_and_zip_splits(row, multi_columns)
    static_data = {col: row[col] for col in row.index if col not in multi_columns}
    expanded = []
    for idx, values in enumerate(zipped_values):
        new_row = {}
        if idx == 0:
            new_row.update(static_data)
        for col, val in zip(multi_columns, values):
            new_row[col] = val
        expanded.append(new_row)
    return expanded


def normalize_and_zip_splits(row, columns):
    lists = [split_entry(row[col]) for col in columns]
    max_len = max(len(lst) for lst in lists)
    padded_lists = [lst + [""] * (max_len - len(lst)) for lst in lists]
    return list(zip(*padded_lists))


def split_entry(value):
    if pd.isna(value):
        return [""]
    return [item.strip() for item in str(value).split("; ")]


def column_dropper(dataframe: pd.DataFrame):
    drop_true = ['LOKASI BONGKAR','JENIS BARANG','JENIS MUATAN']

    for col in dataframe.columns:
        if col in drop_true:
            dataframe = dataframe.drop(columns=col)

    return dataframe


def column_renamer(dataframe: pd.DataFrame, name:str):
    if name in ['domestic','export','empty'] or 'load' in name:
        dataframe = dataframe.rename(columns={'TUJUAN': 'TUJUAN MUAT', 'PEMILIK / AGEN': 'KEAGENAN'})
    elif 'tkii' in name:
        dataframe = dataframe.rename(columns={'PEMILIK / AGEN': 'KEAGENAN'})

    # print(name, dataframe.columns)
    return dataframe


def nat_error_fixer(dataframe, output: list):
    for dtype, column in zip(dataframe.dtypes, dataframe.columns):
        if str(dtype) == 'datetime64[ns]':
            dataframe[column] = dataframe[column].astype(object).where(dataframe[column].notnull(), ' ')

    return dataframe[output]

# Editorial

In [4]:
def ignore_nan(worksheet, row, col, number, cell_format=None):
    if math.isnan(number):
        return worksheet.write_blank(row, col, None, cell_format)
    else:
        # Return control to the calling write() method for any other number.
        return None


def create_header_formats(output_name: str, output_format: dict, workbook):
    cell_formats = {'title':[],'sub_title':[],'header':[],'sub_header':[],'filler':[]}
    if output_name == 'clearance':
        cell_formats['title'] = [workbook.add_format(output_format['title_bold']),workbook.add_format(output_format['title_bold'])]
        cell_formats['sub_title'] = [workbook.add_format(output_format['sub_title_left']),workbook.add_format(output_format['sub_title_left']),
                                     workbook.add_format(output_format['sub_title_left']),workbook.add_format(output_format['sub_title_left']),
                                     workbook.add_format(output_format['sub_title_center'])]
        cell_formats['header'] = [workbook.add_format(output_format['left_header']),workbook.add_format(output_format['middle_header']),
                                  workbook.add_format(output_format['middle_header']),workbook.add_format(output_format['middle_header']),
                                  workbook.add_format(output_format['merged_header']),workbook.add_format(output_format['middle_header']),
                                  workbook.add_format(output_format['middle_header']),workbook.add_format(output_format['middle_header']),
                                  workbook.add_format(output_format['middle_header']),workbook.add_format(output_format['middle_header']),
                                  workbook.add_format(output_format['middle_header']),workbook.add_format(output_format['merged_header']),
                                  workbook.add_format(output_format['merged_header']),workbook.add_format(output_format['merged_header']),
                                  workbook.add_format(output_format['right_header'])]
        cell_formats['sub_header'] = [workbook.add_format(output_format['sub_header']),workbook.add_format(output_format['sub_header']),
                                      workbook.add_format(output_format['sub_header']),workbook.add_format(output_format['sub_header']),
                                      workbook.add_format(output_format['sub_header']),workbook.add_format(output_format['sub_header']),
                                      workbook.add_format(output_format['sub_header']),workbook.add_format(output_format['sub_header'])]
        cell_formats['filler'] = [workbook.add_format(output_format['left_filler']),workbook.add_format(output_format['middle_filler']),
                                  workbook.add_format(output_format['middle_filler']),workbook.add_format(output_format['middle_filler']),
                                  workbook.add_format(output_format['middle_filler']),workbook.add_format(output_format['middle_filler']),
                                  workbook.add_format(output_format['middle_filler']),workbook.add_format(output_format['middle_filler']),
                                  workbook.add_format(output_format['middle_filler']),workbook.add_format(output_format['middle_filler']),
                                  workbook.add_format(output_format['middle_filler']),workbook.add_format(output_format['middle_filler']),
                                  workbook.add_format(output_format['middle_filler']),workbook.add_format(output_format['middle_filler']),
                                  workbook.add_format(output_format['middle_filler']),workbook.add_format(output_format['middle_filler']),
                                  workbook.add_format(output_format['middle_filler']),workbook.add_format(output_format['middle_filler']),
                                  workbook.add_format(output_format['middle_filler']),workbook.add_format(output_format['right_filler'])]

    elif 'sib' in output_name:
        cell_formats['title'] = [workbook.add_format(output_format['title_bold'])]
        cell_formats['sub_title'] = [workbook.add_format(output_format['sub_title_center']),workbook.add_format(output_format['sub_title_center']),
                                     workbook.add_format(output_format['sub_title_center']),workbook.add_format(output_format['sub_title_center'])]
        cell_formats['header'] = [workbook.add_format(output_format['left_header']),workbook.add_format(output_format['merged_header']),
                                  workbook.add_format(output_format['middle_header']),workbook.add_format(output_format['middle_header']),
                                  workbook.add_format(output_format['middle_header']),workbook.add_format(output_format['middle_header']),
                                  workbook.add_format(output_format['merged_header']),workbook.add_format(output_format['merged_header']),
                                  workbook.add_format(output_format['middle_header']),workbook.add_format(output_format['right_header'])]
        cell_formats['sub_header'] = [workbook.add_format(output_format['sub_header']),workbook.add_format(output_format['sub_header']),
                                      workbook.add_format(output_format['sub_header']),workbook.add_format(output_format['sub_header']),
                                      workbook.add_format(output_format['sub_header']),workbook.add_format(output_format['sub_header']),
                                      workbook.add_format(output_format['sub_header']),workbook.add_format(output_format['sub_header']),
                                      workbook.add_format(output_format['sub_header'])]
        cell_formats['filler'] = [workbook.add_format(output_format['left_filler']),workbook.add_format(output_format['middle_filler']),
                                  workbook.add_format(output_format['middle_filler']),workbook.add_format(output_format['middle_filler']),
                                  workbook.add_format(output_format['middle_filler']),workbook.add_format(output_format['middle_filler']),
                                  workbook.add_format(output_format['middle_filler']),workbook.add_format(output_format['middle_filler']),
                                  workbook.add_format(output_format['middle_filler']),workbook.add_format(output_format['middle_filler']),
                                  workbook.add_format(output_format['middle_filler']),workbook.add_format(output_format['middle_filler']),
                                  workbook.add_format(output_format['middle_filler']),workbook.add_format(output_format['middle_filler']),
                                  workbook.add_format(output_format['middle_filler']),workbook.add_format(output_format['right_filler'])]

    elif 'tkii' in output_name:
        cell_formats['title'] = [workbook.add_format(output_format['title_bold']),workbook.add_format(output_format['title_bold'])]
        cell_formats['sub_title'] = [workbook.add_format(output_format['sub_title_center']),workbook.add_format(output_format['sub_title_center']),
                                     workbook.add_format(output_format['sub_title_center']),workbook.add_format(output_format['sub_title_right']),
                                     workbook.add_format(output_format['sub_title_center']),workbook.add_format(output_format['sub_title_left']),
                                     workbook.add_format(output_format['sub_title_right']),workbook.add_format(output_format['sub_title_center']),
                                     workbook.add_format(output_format['sub_title_left']),workbook.add_format(output_format['sub_title_center']),
                                     workbook.add_format(output_format['sub_title_center'])]
        cell_formats['header'] = [workbook.add_format(output_format['left_header']),workbook.add_format(output_format['merged_header']),
                                  workbook.add_format(output_format['middle_header']),workbook.add_format(output_format['merged_header']),
                                  workbook.add_format(output_format['middle_header']),workbook.add_format(output_format['merged_header']),
                                  workbook.add_format(output_format['merged_header']),workbook.add_format(output_format['merged_header']),
                                  workbook.add_format(output_format['right_header'])]
        cell_formats['sub_header'] = [workbook.add_format(output_format['sub_header']),workbook.add_format(output_format['sub_header']),
                                      workbook.add_format(output_format['sub_header']),workbook.add_format(output_format['sub_header']),
                                      workbook.add_format(output_format['sub_header']),workbook.add_format(output_format['sub_header']),
                                      workbook.add_format(output_format['sub_header']),workbook.add_format(output_format['sub_header']),
                                      workbook.add_format(output_format['sub_header']),workbook.add_format(output_format['sub_header']),
                                      workbook.add_format(output_format['sub_header']),workbook.add_format(output_format['sub_header']),
                                      workbook.add_format(output_format['sub_header'])]
        cell_formats['filler'] = [workbook.add_format(output_format['left_filler']),workbook.add_format(output_format['middle_filler']),
                                  workbook.add_format(output_format['middle_filler']),workbook.add_format(output_format['middle_filler']),
                                  workbook.add_format(output_format['middle_filler']),workbook.add_format(output_format['middle_filler']),
                                  workbook.add_format(output_format['middle_filler']),workbook.add_format(output_format['middle_filler']),
                                  workbook.add_format(output_format['middle_filler']),workbook.add_format(output_format['middle_filler']),
                                  workbook.add_format(output_format['middle_filler']),workbook.add_format(output_format['middle_filler']),
                                  workbook.add_format(output_format['middle_filler']),workbook.add_format(output_format['middle_filler']),
                                  workbook.add_format(output_format['middle_filler']),workbook.add_format(output_format['middle_filler']),
                                  workbook.add_format(output_format['middle_filler']),workbook.add_format(output_format['middle_filler']),
                                  workbook.add_format(output_format['middle_filler']),workbook.add_format(output_format['middle_filler']),
                                  workbook.add_format(output_format['middle_filler']),workbook.add_format(output_format['middle_filler']),
                                  workbook.add_format(output_format['right_filler'])]
    elif 'domestic' in output_name:
        cell_formats['title'] = [workbook.add_format(output_format['title_bold']),workbook.add_format(output_format['title_bold'])]
        cell_formats['sub_title'] = [workbook.add_format(output_format['sub_title_center'])]
        cell_formats['header'] = [workbook.add_format(output_format['left_header']),workbook.add_format(output_format['merged_header']),
                                  workbook.add_format(output_format['middle_header']),workbook.add_format(output_format['middle_header']),
                                  workbook.add_format(output_format['middle_header']),workbook.add_format(output_format['middle_header']),
                                  workbook.add_format(output_format['middle_header']),workbook.add_format(output_format['merged_header']),
                                  workbook.add_format(output_format['merged_header_right'])]
        cell_formats['sub_header'] = [workbook.add_format(output_format['sub_header']),workbook.add_format(output_format['sub_header']),
                                      workbook.add_format(output_format['sub_header']),workbook.add_format(output_format['sub_header']),
                                      workbook.add_format(output_format['sub_header']),workbook.add_format(output_format['sub_header']),
                                      workbook.add_format(output_format['sub_header']),workbook.add_format(output_format['sub_header_right'])]
        cell_formats['filler'] = [workbook.add_format(output_format['left_filler']),workbook.add_format(output_format['middle_filler']),
                                  workbook.add_format(output_format['middle_filler']),workbook.add_format(output_format['middle_filler']),
                                  workbook.add_format(output_format['middle_filler']),workbook.add_format(output_format['middle_filler']),
                                  workbook.add_format(output_format['middle_filler']),workbook.add_format(output_format['middle_filler']),
                                  workbook.add_format(output_format['middle_filler']),workbook.add_format(output_format['middle_filler']),
                                  workbook.add_format(output_format['middle_filler']),workbook.add_format(output_format['middle_filler']),
                                  workbook.add_format(output_format['middle_filler']),workbook.add_format(output_format['middle_filler']),
                                  workbook.add_format(output_format['middle_filler']),workbook.add_format(output_format['right_filler'])]

    elif 'export' in output_name:
        cell_formats['title'] = [workbook.add_format(output_format['title_bold']),workbook.add_format(output_format['title_bold'])]
        cell_formats['sub_title'] = [workbook.add_format(output_format['sub_title_center'])]
        cell_formats['header'] = [workbook.add_format(output_format['left_header']),workbook.add_format(output_format['merged_header']),
                                  workbook.add_format(output_format['middle_header']),workbook.add_format(output_format['middle_header']),
                                  workbook.add_format(output_format['middle_header']),workbook.add_format(output_format['middle_header']),
                                  workbook.add_format(output_format['middle_header']),workbook.add_format(output_format['merged_header']),
                                  workbook.add_format(output_format['merged_header']),workbook.add_format(output_format['right_header'])]
        cell_formats['sub_header'] = [workbook.add_format(output_format['sub_header']),workbook.add_format(output_format['sub_header']),
                                      workbook.add_format(output_format['sub_header']),workbook.add_format(output_format['sub_header']),
                                      workbook.add_format(output_format['sub_header']),workbook.add_format(output_format['sub_header']),
                                      workbook.add_format(output_format['sub_header']),workbook.add_format(output_format['sub_header'])]
        cell_formats['filler'] = [workbook.add_format(output_format['left_filler']),workbook.add_format(output_format['middle_filler']),
                                  workbook.add_format(output_format['middle_filler']),workbook.add_format(output_format['middle_filler']),
                                  workbook.add_format(output_format['middle_filler']),workbook.add_format(output_format['middle_filler']),
                                  workbook.add_format(output_format['middle_filler']),workbook.add_format(output_format['middle_filler']),
                                  workbook.add_format(output_format['middle_filler']),workbook.add_format(output_format['middle_filler']),
                                  workbook.add_format(output_format['middle_filler']),workbook.add_format(output_format['middle_filler']),
                                  workbook.add_format(output_format['middle_filler']),workbook.add_format(output_format['middle_filler']),
                                  workbook.add_format(output_format['middle_filler']),workbook.add_format(output_format['middle_filler']),
                                  workbook.add_format(output_format['right_filler'])]

    elif 'empty' in output_name:
        cell_formats['title'] = [workbook.add_format(output_format['title_bold']),workbook.add_format(output_format['title_bold'])]
        cell_formats['sub_title'] = [workbook.add_format(output_format['sub_title_center'])]
        cell_formats['header'] = [workbook.add_format(output_format['left_header']),workbook.add_format(output_format['merged_header']),
                                  workbook.add_format(output_format['middle_header']),workbook.add_format(output_format['merged_header']),
                                  workbook.add_format(output_format['merged_header']),workbook.add_format(output_format['merged_header_right'])]
        cell_formats['sub_header'] = [workbook.add_format(output_format['sub_header']),workbook.add_format(output_format['sub_header']),
                                      workbook.add_format(output_format['sub_header']),workbook.add_format(output_format['sub_header']),
                                      workbook.add_format(output_format['sub_header']),workbook.add_format(output_format['sub_header']),
                                      workbook.add_format(output_format['sub_header']),workbook.add_format(output_format['sub_header']),
                                      workbook.add_format(output_format['sub_header_right'])]
        cell_formats['filler'] = [workbook.add_format(output_format['left_filler']),workbook.add_format(output_format['middle_filler']),
                                  workbook.add_format(output_format['middle_filler']),workbook.add_format(output_format['middle_filler']),
                                  workbook.add_format(output_format['middle_filler']),workbook.add_format(output_format['middle_filler']),
                                  workbook.add_format(output_format['middle_filler']),workbook.add_format(output_format['middle_filler']),
                                  workbook.add_format(output_format['middle_filler']),workbook.add_format(output_format['right_filler'])]

    elif 'load' in output_name:
        cell_formats['title'] = [workbook.add_format(output_format['title_bold']),workbook.add_format(output_format['title_bold'])]
        cell_formats['sub_title'] = [workbook.add_format(output_format['sub_title_center'])]
        cell_formats['header'] = [workbook.add_format(output_format['left_header']),workbook.add_format(output_format['merged_header']),
                                  workbook.add_format(output_format['middle_header']),workbook.add_format(output_format['middle_header']),
                                  workbook.add_format(output_format['middle_header']),workbook.add_format(output_format['middle_header']),
                                  workbook.add_format(output_format['middle_header']),workbook.add_format(output_format['merged_header']),
                                  workbook.add_format(output_format['merged_header_right'])]
        cell_formats['sub_header'] = [workbook.add_format(output_format['sub_header']),workbook.add_format(output_format['sub_header']),
                                      workbook.add_format(output_format['sub_header']),workbook.add_format(output_format['sub_header']),
                                      workbook.add_format(output_format['sub_header']),workbook.add_format(output_format['sub_header']),
                                      workbook.add_format(output_format['sub_header']),workbook.add_format(output_format['sub_header_right'])]
        cell_formats['filler'] = [workbook.add_format(output_format['left_filler']),workbook.add_format(output_format['middle_filler']),
                                  workbook.add_format(output_format['middle_filler']),workbook.add_format(output_format['middle_filler']),
                                  workbook.add_format(output_format['middle_filler']),workbook.add_format(output_format['middle_filler']),
                                  workbook.add_format(output_format['middle_filler']),workbook.add_format(output_format['middle_filler']),
                                  workbook.add_format(output_format['middle_filler']),workbook.add_format(output_format['middle_filler']),
                                  workbook.add_format(output_format['middle_filler']),workbook.add_format(output_format['middle_filler']),
                                  workbook.add_format(output_format['middle_filler']),workbook.add_format(output_format['middle_filler']),
                                  workbook.add_format(output_format['middle_filler']),workbook.add_format(output_format['right_filler'])]

    return cell_formats


def create_main_data_formats(output_format: dict, workbook):
    cell_formats = {'most_left':[],'most_right':[],'dates':[],'left_aligned':[],'other_middle':[]}
    cell_formats['most_left'] = [workbook.add_format(output_format['left_upper_main']),
                                 workbook.add_format(output_format['left_downer_main']),
                                 workbook.add_format(output_format['left_mid_main'])]
    cell_formats['most_right'] = [workbook.add_format(output_format['right_upper_main']),
                                  workbook.add_format(output_format['right_downer_main']),
                                  workbook.add_format(output_format['right_mid_main'])]
    cell_formats['center_right'] = [workbook.add_format(output_format['center_upper_main']),
                                  workbook.add_format(output_format['center_downer_main']),
                                  workbook.add_format(output_format['center_mid_main'])]
    cell_formats['dates'] = [workbook.add_format(output_format['upper_dates']),
                             workbook.add_format(output_format['downer_dates']),
                             workbook.add_format(output_format['middle_dates'])]
    cell_formats['left_aligned'] = [workbook.add_format(output_format['upper_left_aligning']),
                                    workbook.add_format(output_format['downer_left_aligning']),
                                    workbook.add_format(output_format['middle_left_aligning'])]
    cell_formats['other_middle'] = [workbook.add_format(output_format['middle_upper_main']),
                                    workbook.add_format(output_format['middle_downer_main']),
                                    workbook.add_format(output_format['middle_mid_main'])]
    cell_formats['for_nihil'] = workbook.add_format(output_format['for_nihil'])
    cell_formats['summary'] = [workbook.add_format(output_format['load_names']),
                               workbook.add_format(output_format['total_number'])]

    return cell_formats


def write_header(header_parts, cell_format, header_detail, worksheet):
    for key in header_parts:
        # print(len(header_detail['ranges'][key]))
        for item in range(len(header_detail['ranges'][key])):
            # print(key, item, header_detail['ranges'][key][item], header_detail['texts'][key][item], cell_format[key][item])
            if ':' in header_detail['ranges'][key][item]:
                # print(key,item,header_detail['ranges'][key][item])
                worksheet.merge_range(header_detail['ranges'][key][item],header_detail['texts'][key][item],cell_format[key][item])
            else:
                worksheet.write(header_detail['ranges'][key][item],header_detail['texts'][key][item],cell_format[key][item])


def write_main_data(output_property: dict, worksheet):
    main_data = output_property['data']
    len_data = output_property['len_data']
    cell_format = output_property['format']
    start_row = output_property['start_row']-1
    init=0
    for key in main_data.keys():
        for num in range(len_data):
            if key == 'NO':
                if num == 0:
                    worksheet.write(num+start_row,init,main_data[key][num],cell_format['most_left'][0])
                elif num == len(main_data[key])-1:
                    worksheet.write(num+start_row,init,main_data[key][num],cell_format['most_left'][1])
                elif num != 0 and num != len(main_data[key])-1:
                    worksheet.write(num+start_row,init,main_data[key][num],cell_format['most_left'][2])
            elif key in ['WAKTU TIBA', 'WAKTU TOLAK']:
                if num == 0:
                    worksheet.write(num+start_row,init,main_data[key][num],cell_format['dates'][0])
                elif num == len(main_data[key])-1:
                    worksheet.write(num+start_row,init,main_data[key][num],cell_format['dates'][1])
                elif num != 0 and num != len(main_data[key])-1:
                    worksheet.write(num+start_row,init,main_data[key][num],cell_format['dates'][2])
            elif key in ['NAMA KAPAL', 'GT','NT','TANDA SELAR','KEAGENAN']:
                if num == 0:
                    worksheet.write(num+start_row,init,main_data[key][num],cell_format['left_aligned'][0])
                elif num == len(main_data[key])-1:
                    worksheet.write(num+start_row,init,main_data[key][num],cell_format['left_aligned'][1])
                elif num != 0 and num != len(main_data[key])-1:
                    worksheet.write(num+start_row,init,main_data[key][num],cell_format['left_aligned'][2])
            elif key in ['PEMILIK / AGEN','KATEGORI','KET.']:
                if num == 0:
                    worksheet.write(num+start_row,init,main_data[key][num],cell_format['most_right'][0])
                elif num == len(main_data[key])-1:
                    worksheet.write(num+start_row,init,main_data[key][num],cell_format['most_right'][1])
                elif num != 0 and num != len(main_data[key])-1:
                    worksheet.write(num+start_row,init,main_data[key][num],cell_format['most_right'][2])
            elif key in ['TUJUAN MUAT']:
                if num == 0:
                    worksheet.write(num+start_row,init,main_data[key][num],cell_format['center_right'][0])
                elif num == len(main_data[key])-1:
                    worksheet.write(num+start_row,init,main_data[key][num],cell_format['center_right'][1])
                elif num != 0 and num != len(main_data[key])-1:
                    worksheet.write(num+start_row,init,main_data[key][num],cell_format['center_right'][2])
            else:
                if num == 0:
                    worksheet.write(num+start_row,init,main_data[key][num],cell_format['other_middle'][0])
                elif num == len(main_data[key])-1:
                    worksheet.write(num+start_row,init,main_data[key][num],cell_format['other_middle'][1])
                elif num != 0 and num != len(main_data[key])-1:
                    worksheet.write(num+start_row,init,main_data[key][num],cell_format['other_middle'][2])
        init += 1


def write_empty_main_data(output_property: dict, worksheet):
    start_row = output_property['start_row']-1
    start_col = 0
    last_row = output_property['start_row']-1
    last_col = len(output_property['col'])-1
    cell_format = output_property['format']
    # print(start_row, start_col, last_row, last_col,cell_format['for_nihil'])
    worksheet.merge_range(start_row, start_col,
                          last_row, last_col,
                          'N I H I L',
                          cell_format['for_nihil'])


def set_pixels(pixel_size: dict, worksheet):
    worksheet.set_default_row(pixel_size['default_height'])

    set_worksheet_row(pixel_size['row'], worksheet)
    set_worksheet_col(pixel_size['column'], worksheet)


def set_worksheet_row(pixel_size, worksheet):
    for index,item in enumerate(pixel_size):
        worksheet.set_row(index, item)


def set_worksheet_col(pixel_size, worksheet):
    for index,item in enumerate(pixel_size):
        worksheet.set_column(index,index,item)


def create_output(output_model: list, writer: pd.ExcelWriter, dictionary: dict, row: int):
    output_dict = output_model[0]
    output_name = output_model[1]
    output_format = output_model[2]
    output_cell = output_model[3]
    output_sheet = output_model[4]
    output_start = output_model[5]

    xlworkbook= writer.book

    # print(output_name)
    # if output_name == 'clearance':
    worksheet = xlworkbook.add_worksheet(output_sheet)

    # Add the write() handler/callback to the worksheet if it finds NAN/INF
    worksheet.add_write_handler(float, ignore_nan)

    header_format = create_header_formats(output_name, output_format, xlworkbook)
    main_data_format = create_main_data_formats(output_format, xlworkbook)

    output_subdict = list(output_dict['ranges'].keys())

    write_header(output_subdict, header_format, output_dict, worksheet)
    if len(dictionary) == 0:
        write_empty_main_data({'start_row': output_start, 'col': output_cell['column'],'format': main_data_format}, worksheet)
        worksheet.set_row(output_start-1, 100)
        set_worksheet_col(output_cell['column'], worksheet)
    else:
        write_main_data({'data': dictionary, 'len_data': row, 'format': main_data_format, 'start_row': output_start}, worksheet)
        set_pixels(output_cell, worksheet)

# Main

In [8]:
# global variables declaration
data_address = 'data/'
support_address = 'support/'
xlsx_in = '00_raw_data.xlsx'
xlsx_out = '02 Draft Laporan.xlsx'
json01 = 'no_01_columns.json'
json02 = 'no_02_summary_vars.json'
json03 = 'no_03_output_vars.json'
json04 = 'no_04_sheet_formats.json'
filenames = {'columns': json01, 'summaries': json02, 'output_vars': json03, 'sheet_format': json04}
# -----------------------------------------------------------------------------
# set global variables here
jsons = {'columns': [], 'summaries': [], 'output_vars': [], 'sheet_format': []}

for key in jsons.keys():
    jsons[key] = read_jsons(filename=filenames[key], address=support_address)
    #print(jsons[key])
# -----------------------------------------------------------------------------

def main():
    # Step 01: Initiate Write Engine to save the output
    excel_file = pd.ExcelWriter(data_address+xlsx_out, engine='xlsxwriter', engine_kwargs={"options": {"nan_inf_to_errors": True}})
    # Step 02: Draw data from xlsx file
    df_raw = read_excel_file(xlsx_in, data_address)
    # print(df_raw.columns)
    # Step 03: Reset Row Numbers in Dataframe
    df_preparing = reset_row_numbers(df_raw, 'NO', len(df_raw), True)
    # This is for Step 04 to 08
    output_names = list(jsons['output_vars'].keys())
    # print(output_names)
    for name in output_names:
        # Step 04: Column extraction
        # print(jsons['columns']['initiate'][name])
        df_init = extract_columns(df_preparing, jsons['columns']['initiate'][name])
        # Step 05: Customize columns, by adding or organizing columns
        # print(jsons['output_vars'][name]['final_variables'])
        df_custom = customize_columns(df_init, name, jsons['columns']['customize'][name],
                                          jsons['output_vars'][name]['final_variables'])
        # Step 06: Filter row by certain column(s)
        df_filtered = filter_row(df_custom, name, jsons['columns']['filter'][name])
        # Step 07: Rearrange rows that contains char ";"
        property = {'name': name, 'keys': jsons['columns']['rearrange'][name]}
        dict_final, len_dict = rearrange_rows(df_filtered, name, property['keys'],
                                                      jsons['output_vars'][name]['final_variables'])
        # Step 08: Write and save an output
        output_detail = [jsons['output_vars'][name],
                        name,
                        jsons['sheet_format'],
                        jsons['output_vars'][name]['pixel_size'],
                        jsons['output_vars'][name]['sheet_name'],
                        jsons['output_vars'][name]['start_row']]
        create_output(output_detail, excel_file, dict_final, len_dict)
        # break
    excel_file.close()
main()